# Thí nghiệm thử: WeedsGalore, phân vùng cỏ dại và quyết định phun thuốc

Mục đích: kiểm tra đề tài có khả thi không trước khi đầu tư tiếp. Thí nghiệm gồm ba phần:
1. Huấn luyện DeepLabv3+ dùng ảnh 5 dải phổ, bài toán 3 lớp (nền, ngô, cỏ), theo thiết lập của Celikkan và cộng sự (WACV 2025).
2. Đánh giá trên tập test và so với kết quả trong bài báo (mIoU 82,90%).
3. Tầng quyết định: chia mỗi ô ảnh thành các ô lưới, quyết định **phun / không phun / chưa quyết** bằng conformal. Phương án chính hiệu chỉnh chéo 2 phần theo vị trí trên tập test (4 vị trí hiệu chỉnh cho 4 vị trí còn lại và ngược lại), vì tập val đã dùng để chọn mô hình; hiệu chỉnh trên val là phương án phụ.

**Lần chạy 2:** batch 8 như bài báo (lần 1 dùng batch 2, là mặc định trong code của tác giả), mọi thiết lập khác giữ nguyên lần 1. Kết quả lưu vào thư mục mới `run_dlv3p_msi_seed0_b8`, không ghi đè lần 1 (`run_dlv3p_msi_seed0`). Dữ liệu gọn của lần 1 trên Drive được dùng lại, nên mục 2 sẽ tự bỏ qua.

**Tiêu chí đạt/không đạt** (đề xuất, sửa được trong `src/weeds/analyze.py`): mIoU test ≥ 77,90; lượng thuốc giảm ≥ 30% so với phun toàn ruộng; tỉ lệ ô "chưa quyết" ≤ 30%. Các tiêu chí này tính ở thiết lập chính: ô lưới 0,5 m, ngưỡng cỏ 1% (ngưỡng minh họa), α = β = 0,1, gộp 3 đợt bay trước khi phun thuốc.

**Cách chạy nhanh nhất:** *Runtime → Change runtime type → T4 GPU*, rồi *Runtime → Run all*. Ở mục 1, Colab hỏi quyền truy cập Google Drive: bấm đồng ý. Phần tải và chuẩn bị dữ liệu (mục 2) tốn thêm khoảng 5–10 phút GPU. Nếu muốn tiết kiệm hạn mức GPU, chạy mục 1–2 trên runtime CPU trước, rồi đổi sang T4 và Run all; mục 2 sẽ tự bỏ qua vì dữ liệu đã có trên Drive.

- Nếu một bước bị lỗi, "Run all" sẽ dừng ở bước đó. Hãy gửi thông báo lỗi cho Claude.
- Huấn luyện tự dừng an toàn trước khoảng 3,5 giờ và lưu checkpoint lên Drive. Nếu chưa xong, ở phiên sau chạy lại mục 1, 3 rồi 5 để chạy tiếp.
- Giữ tab Colab mở trong lúc chạy, thỉnh thoảng xem qua, vì Colab có thể ngắt kết nối khi trình duyệt không hoạt động lâu.

Kết quả nằm trong `MyDrive/De_Tai_Nong_San/weeds_pilot/run_dlv3p_msi_seed0_b8/`. Mục cuối tạo file `ket_qua_weeds_pilot_b8.zip` để tải về.

## 1. Kết nối Drive và ghi mã nguồn (chạy ở **mọi** phiên, cả CPU lẫn GPU)

In [ ]:
import os, sys, json
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/De_Tai_Nong_San/weeds_pilot')
DATA_DRIVE = BASE / 'data'
BATCH = 8  # run 2: the paper's batch size; run 1 used 2 (authors' code default) and lives in run_dlv3p_msi_seed0
RUN = BASE / f'run_dlv3p_msi_seed0_b{BATCH}'  # a new folder, so run 1's train_done.json cannot skip training
RESULTS = Path(f'results/weeds_pilot_b{BATCH}')  # folder inside the downloaded zip; run 1 is in results/weeds_pilot
for p in (DATA_DRIVE, RUN):
    p.mkdir(parents=True, exist_ok=True)

WORK = Path('/content/weeds_pilot')
FILES = {"src/__init__.py": "", "src/weeds/__init__.py": "\"\"\"Pilot on WeedsGalore (Celikkan et al., WACV 2025): crop/weed segmentation and a spraying decision layer.\"\"\"\n", "src/weeds/data.py": "\"\"\"Read the WeedsGalore dataset zip and convert each split into compact arrays.\n\nBand order R, G, B, NIR, RE and the 3-class merge (0 background, 1 maize, 2 any weed) follow the authors' loader\n(github.com/GFZ/weedsgalore, src/datasets/weedsgalore/weedsgalore.py). Images stay uint16; dividing by 65535 gives the\n[0, 1] scaling that their matplotlib-based reader produces.\n\"\"\"\nimport io\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\n\nROOT = 'weedsgalore-dataset'\nBANDS = ('R', 'G', 'B', 'NIR', 'RE')\nSPLITS = ('train', 'val', 'test')\nDATES = ('2023-05-25', '2023-05-30', '2023-06-06', '2023-06-15')\nPRE_HERBICIDE_DATES = DATES[:3]  # herbicide was applied on 12 June 2023 (Celikkan et al. 2025, 3.1)\nEXPECTED_TILES = {'train': 104, 'val': 26, 'test': 26}\nEXPECTED_TEST_LOCATIONS = 8  # 2 field patches x 4 sampled points (Celikkan et al. 2025, 3.2-3.3)\nTILE = 600\nSEMANTIC_CLASSES = ('background', 'maize', 'amaranth', 'barnyard_grass', 'quickweed', 'weed_other')\nCLASSES3 = ('background', 'crop', 'weed')\n# Celikkan et al. 2025, Table 3: DeepLabv3+ with 5-band (MSI) input, 3-class setting, test IoU in %\nPAPER_TEST_3CLASS_MSI = {'miou': 82.90, 'background': 98.45, 'crop': 72.93, 'weed': 77.31}\n\n\ndef split_ids(zf, split):\n    return zf.read(f'{ROOT}/splits/{split}.txt').decode().split()\n\n\ndef tile_date(tile_id):\n    return tile_id[:10]\n\n\ndef read_png(zf, name):\n    return np.array(Image.open(io.BytesIO(zf.read(name))))\n\n\ndef read_tile(zf, tile_id):\n    \"\"\"(5, 600, 600) uint16 image in BANDS order and (600, 600) uint8 semantic mask with values 0-5.\"\"\"\n    base = f'{ROOT}/{tile_date(tile_id)}'\n    bands = [read_png(zf, f'{base}/images/{tile_id}_{b}.png') for b in BANDS]\n    for b, a in zip(BANDS, bands):\n        if a.dtype != np.uint16:  # checked before stacking: an 8-bit band would otherwise be scaled by 65535 silently\n            raise ValueError(f'{tile_id} band {b}: expected 16-bit PNG, got {a.dtype}')\n    mask = read_png(zf, f'{base}/semantics/{tile_id}.png')\n    return np.stack(bands), mask.astype(np.uint8)\n\n\ndef location_blocks(ids):\n    \"\"\"Group the tile ids of a split file into field locations. In the val and test files each location is written\n    as a block of consecutive ids with decreasing dates (checked visually for the test split on 2026-10-29: shared\n    field markers and tyre tracks across dates). Returns a list of blocks.\"\"\"\n    blocks, cur = [], []\n    for t in ids:\n        if cur and tile_date(t) >= tile_date(cur[-1]):\n            blocks.append(cur)\n            cur = []\n        cur.append(t)\n    if cur:\n        blocks.append(cur)\n    return blocks\n\n\ndef to_3class(mask):\n    \"\"\"0 background, 1 maize, 2 weed (all weed species merged), as in the authors' binary_label.\"\"\"\n    out = mask.copy()\n    out[out > 1] = 2\n    return out\n\n\ndef load_split(zip_path, split):\n    with zipfile.ZipFile(zip_path) as zf:\n        ids = split_ids(zf, split)\n        tiles = [read_tile(zf, t) for t in ids]\n    x = np.stack([t[0] for t in tiles])\n    y = np.stack([t[1] for t in tiles])\n    check_split(split, ids, x, y)\n    return {'ids': np.array(ids), 'x': x, 'y': y}\n\n\ndef check_split(split, ids, x, y):\n    n = EXPECTED_TILES[split]\n    if len(ids) != n or x.shape != (n, len(BANDS), TILE, TILE) or y.shape != (n, TILE, TILE):\n        raise ValueError(f'{split}: unexpected shapes ids={len(ids)} x={x.shape} y={y.shape}')\n    if x.dtype != np.uint16 or y.max() > len(SEMANTIC_CLASSES) - 1:\n        raise ValueError(f'{split}: unexpected dtype {x.dtype} or label values up to {y.max()}')\n    if not set(tile_date(t) for t in ids) <= set(DATES):\n        raise ValueError(f'{split}: unknown acquisition date in ids')\n\n\ndef build_npz(zip_path, out_dir):\n    \"\"\"Write weedsgalore_<split>.npz (ids, x uint16, y uint8 semantic) for each split; returns the paths.\"\"\"\n    from pathlib import Path\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    paths = {}\n    for split in SPLITS:\n        d = load_split(zip_path, split)\n        paths[split] = out_dir / f'weedsgalore_{split}.npz'\n        np.savez_compressed(paths[split], **d)\n        print(f'{split}: {len(d[\"ids\"])} tiles, x {d[\"x\"].shape} {d[\"x\"].dtype}, saved {paths[split]}')\n    return paths\n\n\ndef load_npz(path):\n    with np.load(path) as d:\n        return {'ids': d['ids'].astype(str), 'x': d['x'], 'y': d['y']}\n", "src/weeds/model.py": "\"\"\"DeepLabv3+ with a ResNet-50 backbone, output stride 8, as used by the WeedsGalore authors\n(their code: VainF/DeepLabV3Plus-Pytorch). The ImageNet backbone comes from torchvision; the first convolution is\nreplaced by a randomly initialised 5-channel convolution for multispectral input, as the authors did.\nNeeds PyTorch (Colab).\"\"\"\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom torchvision.models import ResNet50_Weights, resnet50\n\n\nclass ASPPConv(nn.Sequential):\n    def __init__(self, cin, cout, dilation):\n        super().__init__(nn.Conv2d(cin, cout, 3, padding=dilation, dilation=dilation, bias=False),\n                         nn.BatchNorm2d(cout), nn.ReLU(inplace=True))\n\n\nclass ASPPPooling(nn.Sequential):\n    def __init__(self, cin, cout):\n        super().__init__(nn.AdaptiveAvgPool2d(1), nn.Conv2d(cin, cout, 1, bias=False),\n                         nn.BatchNorm2d(cout), nn.ReLU(inplace=True))\n\n    def forward(self, x):\n        size = x.shape[-2:]\n        return F.interpolate(super().forward(x), size=size, mode='bilinear', align_corners=False)\n\n\nclass ASPP(nn.Module):\n    def __init__(self, cin, rates, cout=256):\n        super().__init__()\n        branches = [nn.Sequential(nn.Conv2d(cin, cout, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))]\n        branches += [ASPPConv(cin, cout, r) for r in rates]\n        branches.append(ASPPPooling(cin, cout))\n        self.branches = nn.ModuleList(branches)\n        self.project = nn.Sequential(nn.Conv2d(cout * len(branches), cout, 1, bias=False), nn.BatchNorm2d(cout),\n                                     nn.ReLU(inplace=True), nn.Dropout(0.1))\n\n    def forward(self, x):\n        return self.project(torch.cat([b(x) for b in self.branches], dim=1))\n\n\nclass DeepLabV3Plus(nn.Module):\n    def __init__(self, in_channels=5, num_classes=3, pretrained_backbone=True, rates=(12, 24, 36)):\n        super().__init__()\n        weights = ResNet50_Weights.IMAGENET1K_V1 if pretrained_backbone else None\n        r = resnet50(weights=weights, replace_stride_with_dilation=[False, True, True])  # output stride 8\n        if in_channels != 3:\n            r.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)\n        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool)\n        self.layer1, self.layer2, self.layer3, self.layer4 = r.layer1, r.layer2, r.layer3, r.layer4\n        self.low_proj = nn.Sequential(nn.Conv2d(256, 48, 1, bias=False), nn.BatchNorm2d(48), nn.ReLU(inplace=True))\n        self.aspp = ASPP(2048, rates)\n        self.classifier = nn.Sequential(nn.Conv2d(304, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256),\n                                        nn.ReLU(inplace=True), nn.Conv2d(256, num_classes, 1))\n        for m in [*self.low_proj.modules(), *self.aspp.modules(), *self.classifier.modules()]:\n            if isinstance(m, nn.Conv2d):\n                nn.init.kaiming_normal_(m.weight)\n            elif isinstance(m, nn.BatchNorm2d):\n                nn.init.ones_(m.weight)\n                nn.init.zeros_(m.bias)\n\n    def forward(self, x):\n        size = x.shape[-2:]\n        low = self.layer1(self.stem(x))\n        high = self.layer4(self.layer3(self.layer2(low)))\n        out = F.interpolate(self.aspp(high), size=low.shape[-2:], mode='bilinear', align_corners=False)\n        out = self.classifier(torch.cat([self.low_proj(low), out], dim=1))\n        return F.interpolate(out, size=size, mode='bilinear', align_corners=False)\n", "src/weeds/train.py": "\"\"\"Train DeepLabv3+ on WeedsGalore (3 classes, 5-band input), select it on validation mIoU, evaluate on test and save\nper-pixel class probabilities for the decision layer. Settings follow Celikkan et al. 2025 (4.1.1) and their code:\nAdam lr 1e-3, unweighted cross-entropy, rot90 / flip / Gaussian jitter (sd 1/25) augmentation, inputs scaled to [0, 1].\nThe learning rate is constant as in their code; --lr-schedule cosine is an optional variant.\nMixed precision is used to fit 600x600 tiles on a T4. Run on Colab GPU:\n\n  python -m src.weeds.train --data <dir with weedsgalore_<split>.npz> --out <dir> [--stage train|test|all] [--smoke]\n\nTraining resumes from <out>/last.pt and stops safely before --budget-hours.\n\"\"\"\nimport argparse\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom src.weeds.data import CLASSES3, PAPER_TEST_3CLASS_MSI, SEMANTIC_CLASSES, load_npz, to_3class\nfrom src.weeds.model import DeepLabV3Plus\n\n\nclass TileDataset(Dataset):\n    def __init__(self, d, augment):\n        self.x, self.y, self.augment = d['x'], to_3class(d['y']), augment\n\n    def __len__(self):\n        return len(self.x)\n\n    def __getitem__(self, i):\n        img = self.x[i].astype(np.float32) / 65535.0\n        lab = self.y[i]\n        if self.augment:\n            k = np.random.randint(4)\n            img, lab = np.rot90(img, k, axes=(1, 2)), np.rot90(lab, k)\n            flip = np.random.randint(3)  # 0 none, 1 vertical, 2 horizontal\n            if flip == 1:\n                img, lab = img[:, ::-1, :], lab[::-1, :]\n            elif flip == 2:\n                img, lab = img[:, :, ::-1], lab[:, ::-1]\n            img = img + np.random.randn(*img.shape).astype(np.float32) / 25\n        return (torch.from_numpy(np.ascontiguousarray(img)),\n                torch.from_numpy(np.ascontiguousarray(lab).astype(np.int64)))\n\n\ndef seed_worker(worker_id):\n    # each DataLoader worker would otherwise inherit the same numpy RNG state and repeat the same augmentations\n    s = torch.initial_seed() % 2 ** 32\n    np.random.seed(s)\n    random.seed(s)\n\n\ndef confusion(pred, lab, n=3):\n    \"\"\"cm[true, pred] accumulated over all pixels.\"\"\"\n    return torch.bincount((lab * n + pred).flatten(), minlength=n * n).reshape(n, n)\n\n\ndef iou_from_confusion(cm, names=CLASSES3):\n    \"\"\"Per-class IoU and mIoU in %. A class absent from both labels and predictions gets NaN and is left out of the\n    mIoU (torchmetrics, used by the authors, would count it as 0; every class occurs in the test split).\"\"\"\n    cm = cm.double()\n    tp = cm.diag()\n    denom = cm.sum(0) + cm.sum(1) - tp\n    iou = torch.where(denom > 0, tp / denom.clamp(min=1), torch.full_like(tp, float('nan')))\n    res = {c: float(100 * v) for c, v in zip(names, iou)}\n    res['miou'] = float(100 * torch.nanmean(iou))\n    return res\n\n\n@torch.no_grad()\ndef evaluate(model, d, dev, batch=2, keep_probs=False, limit=None, classes=3, bands=None, amp=True):\n    \"\"\"classes: 3 (labels merged by to_3class) or 6 (semantic labels); bands: use only the first `bands` input bands\n    (3 = R, G, B); amp: float16 autocast on GPU as in training, False = float32 as in the authors' evaluate.py.\n    keep_probs (3 classes only) also returns P(crop), P(weed) per pixel in float16.\"\"\"\n    if keep_probs and classes != 3:\n        raise ValueError('probabilities are kept for the 3-class setting only')\n    model.eval()\n    n = len(d['x']) if limit is None else min(limit, len(d['x']))\n    y = to_3class(d['y'][:n]) if classes == 3 else d['y'][:n]\n    cm = torch.zeros(classes, classes, dtype=torch.long, device=dev)\n    probs = []\n    for i in range(0, n, batch):\n        x = torch.from_numpy(d['x'][i:i + batch, :bands].astype(np.float32) / 65535.0).to(dev)\n        lab = torch.from_numpy(y[i:i + batch].astype(np.int64)).to(dev)\n        with torch.autocast('cuda', dtype=torch.float16, enabled=amp and dev.type == 'cuda'):\n            logits = model(x)\n        p = torch.softmax(logits.float(), dim=1)\n        cm += confusion(p.argmax(1), lab, classes)\n        if keep_probs:\n            probs.append(p[:, 1:].half().cpu().numpy())  # P(crop), P(weed)\n    res = iou_from_confusion(cm.cpu(), CLASSES3 if classes == 3 else SEMANTIC_CLASSES)\n    res['confusion'] = cm.cpu().tolist()\n    return (res, np.concatenate(probs)) if keep_probs else res\n\n\ndef torch_load(path, dev):\n    try:\n        return torch.load(path, map_location=dev, weights_only=False)  # our own checkpoint, not third-party\n    except TypeError:\n        return torch.load(path, map_location=dev)\n\n\ndef grad_scaler():\n    try:\n        return torch.amp.GradScaler('cuda')\n    except (AttributeError, TypeError):\n        return torch.cuda.amp.GradScaler()\n\n\ndef train(args, dev):\n    out = Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n    if (out / 'train_done.json').exists():\n        print('training already finished:', (out / 'train_done.json').read_text())\n        return\n    tr, va = load_npz(Path(args.data) / 'weedsgalore_train.npz'), load_npz(Path(args.data) / 'weedsgalore_val.npz')\n    model = DeepLabV3Plus(5, 3, pretrained_backbone=not args.no_pretrained).to(dev)\n    opt = torch.optim.Adam(model.parameters(), lr=args.lr)\n    # 'constant' is the authors' setting; 'cosine' decays the learning rate to zero over args.epochs, one step per epoch\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs) if args.lr_schedule == 'cosine' else None\n    scaler = grad_scaler()\n    start, best, bad, history = 0, -1.0, 0, []\n    if (out / 'last.pt').exists():\n        ck = torch_load(out / 'last.pt', dev)\n        model.load_state_dict(ck['model'])\n        opt.load_state_dict(ck['opt'])\n        scaler.load_state_dict(ck['scaler'])\n        if sched is not None:\n            sched.load_state_dict(ck['sched'])\n        start, best, bad, history = ck['epoch'] + 1, ck['best_miou'], ck['bad'], ck['history']\n        print(f'resumed after epoch {ck[\"epoch\"]} (best val mIoU {best:.2f})')\n    if (out / 'best.json').exists():\n        # best.pt may be newer than last.pt (last.pt is written every few epochs): never let a worse model replace it\n        best = max(best, json.loads((out / 'best.json').read_text())['val_miou'])\n    loader = DataLoader(TileDataset(tr, augment=True), batch_size=args.batch, shuffle=True, drop_last=True,\n                        num_workers=args.workers, worker_init_fn=seed_worker, pin_memory=dev.type == 'cuda',\n                        generator=torch.Generator().manual_seed(args.seed + start))\n    def save_last(epoch):\n        tmp = out / 'last.tmp'\n        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(),\n                    'sched': sched.state_dict() if sched is not None else None,\n                    'epoch': epoch, 'best_miou': best, 'bad': bad, 'history': history}, tmp)\n        tmp.replace(out / 'last.pt')\n\n    t0, finished, reason, epoch = time.time(), False, '', None\n    for epoch in range(start, args.epochs):\n        model.train()\n        te, losses, lr = time.time(), [], opt.param_groups[0]['lr']\n        for it, (x, y) in enumerate(loader):\n            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast('cuda', dtype=torch.float16, enabled=dev.type == 'cuda'):\n                loss = F.cross_entropy(model(x), y)\n            scaler.scale(loss).backward()\n            scaler.step(opt)\n            scaler.update()\n            losses.append(loss.item())\n            if args.max_iters and it + 1 >= args.max_iters:\n                break\n        if sched is not None:\n            sched.step()\n        val = evaluate(model, va, dev, limit=args.limit_val)\n        secs = time.time() - te\n        history.append({'epoch': epoch, 'lr': lr, 'train_loss': float(np.mean(losses)), 'val_miou': val['miou'],\n                        **{f'val_{c}': val[c] for c in CLASSES3}, 'seconds': secs})\n        if val['miou'] > best:\n            best, bad = val['miou'], 0\n            tmp = out / 'best.tmp'\n            torch.save(model.state_dict(), tmp)\n            tmp.replace(out / 'best.pt')\n            (out / 'best.json').write_text(json.dumps({'epoch': epoch, 'val_miou': best}))\n        else:\n            bad += 1\n        if (epoch + 1) % args.ckpt_every == 0:\n            save_last(epoch)\n        (out / 'history.json').write_text(json.dumps(history, indent=1))\n        mem = f' | peak GPU {torch.cuda.max_memory_allocated() / 1e9:.1f} GB' if dev.type == 'cuda' else ''\n        print(f'epoch {epoch:3d} | loss {history[-1][\"train_loss\"]:.4f} | val mIoU {val[\"miou\"]:.2f} '\n              f'(crop {val[\"crop\"]:.1f}, weed {val[\"weed\"]:.1f}) | best {best:.2f} | {secs:.0f} s{mem}', flush=True)\n        if bad >= args.patience:\n            finished, reason = True, f'early stopping: no val improvement for {args.patience} epochs'\n            break\n        if time.time() - t0 + 1.5 * secs > args.budget_hours * 3600:\n            reason = f'time budget of {args.budget_hours} h reached: rerun to resume'\n            break\n    else:\n        finished, reason = True, f'reached {args.epochs} epochs'\n    if epoch is not None:\n        save_last(epoch)  # always checkpoint on exit, whatever stopped the loop\n    print(reason)\n    if finished and not args.smoke:\n        (out / 'train_done.json').write_text(json.dumps({'reason': reason, 'best_val_miou': best,\n                                                         'epochs_run': len(history), 'batch': args.batch,\n                                                         'lr': args.lr, 'seed': args.seed, 'epochs': args.epochs,\n                                                         'patience': args.patience,\n                                                         'lr_schedule': args.lr_schedule}))\n\n\ndef test(args, dev):\n    out = Path(args.out)\n    model = DeepLabV3Plus(5, 3, pretrained_backbone=False).to(dev)\n    model.load_state_dict(torch_load(out / 'best.pt', dev))\n    report = {}\n    for split in ('val', 'test'):\n        d = load_npz(Path(args.data) / f'weedsgalore_{split}.npz')\n        res, probs = evaluate(model, d, dev, keep_probs=True, limit=args.limit_val)\n        n = len(probs)\n        np.savez_compressed(out / f'probs_{split}.npz', ids=d['ids'][:n], p_crop=probs[:, 0], p_weed=probs[:, 1])\n        report[split] = res\n        print(f'{split}: mIoU {res[\"miou\"]:.2f} | background {res[\"background\"]:.2f}, crop {res[\"crop\"]:.2f}, '\n              f'weed {res[\"weed\"]:.2f}')\n    report['paper_test_3class_msi'] = PAPER_TEST_3CLASS_MSI\n    report['test_minus_paper_miou'] = report['test']['miou'] - PAPER_TEST_3CLASS_MSI['miou']\n    (out / 'segmentation_metrics.json').write_text(json.dumps(report, indent=1))\n    print(f'test mIoU {report[\"test\"][\"miou\"]:.2f} vs paper {PAPER_TEST_3CLASS_MSI[\"miou\"]:.2f} '\n          f'(diff {report[\"test_minus_paper_miou\"]:+.2f})')\n\n\ndef parser():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--data', required=True)\n    ap.add_argument('--out', required=True)\n    ap.add_argument('--stage', choices=['train', 'test', 'all'], default='all')\n    ap.add_argument('--epochs', type=int, default=200)\n    ap.add_argument('--patience', type=int, default=30)\n    ap.add_argument('--batch', type=int, default=2, help='authors\\' code default; the paper reports 8 on an A100')\n    ap.add_argument('--ckpt-every', type=int, default=5, help='write last.pt (about 0.5 GB) every N epochs and on exit')\n    ap.add_argument('--lr', type=float, default=1e-3)\n    ap.add_argument('--lr-schedule', choices=['constant', 'cosine'], default='constant',\n                    help='constant as in the authors\\' code; cosine decays to zero over --epochs')\n    ap.add_argument('--workers', type=int, default=2)\n    ap.add_argument('--seed', type=int, default=0)\n    ap.add_argument('--budget-hours', type=float, default=3.5)\n    ap.add_argument('--max-iters', type=int, default=0, help='stop each epoch after this many iterations (smoke test)')\n    ap.add_argument('--limit-val', type=int, default=None, help='evaluate only the first N tiles (smoke test)')\n    ap.add_argument('--smoke', action='store_true', help='1 epoch of 2 iterations, 2 evaluation tiles')\n    ap.add_argument('--no-pretrained', action='store_true', help='random backbone (local tests only; skips download)')\n    return ap\n\n\ndef main():\n    args = parser().parse_args()\n    if args.smoke:\n        args.epochs, args.max_iters, args.limit_val, args.patience = 1, 2, 2, 1\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    torch.backends.cudnn.benchmark = True\n    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n    print('device:', dev, torch.cuda.get_device_name(0) if dev.type == 'cuda' else '')\n    if args.stage in ('train', 'all'):\n        train(args, dev)\n    if args.stage in ('test', 'all'):\n        test(args, dev)\n\n\nif __name__ == '__main__':\n    main()\n", "src/weeds/decision.py": "\"\"\"Spraying decision layer on top of the segmentation output (numpy only, runs anywhere).\n\nEach 600x600 tile is split into square cells of `cell` pixels (200 px = 0.5 m at the 2.5 mm GSD of WeedsGalore).\n  truth  w     : labelled weed fraction of the cell; the cell needs spraying if w >= tau\n  score  s     : mean predicted P(weed) over the cell, i.e. the expected weed fraction\nDecisions per cell: NO_SPRAY, SPRAY or DEFER (scout the cell; scouting is assumed to reveal the true need).\n\nClass-conditional split conformal thresholds, calibrated on held-out cells that played no part in training or model\nselection (Vovk et al.; as used per class by Melki et al. 2025 for weeding), give for an exchangeable new cell\n  P(s < lam_lo | need)   <= alpha   -> NO_SPRAY only when s < lam_lo   (missed spraying is bounded)\n  P(s > lam_hi | no need) <= beta   -> SPRAY only when s > lam_hi      (unneeded spraying is bounded)\nand cells in between are deferred. Cells of one tile are spatially correlated, so exchangeability, and with it the\nguarantee, holds only approximately; the empirical rates are reported next to alpha and beta.\n\"\"\"\nimport math\n\nimport numpy as np\nimport pandas as pd\n\nNO_SPRAY, SPRAY, DEFER = 0, 1, 2\n_EPS = 1e-9  # guards floor/ceil against floating-point error in alpha * (n + 1)\n\n\ndef cell_view(a, cell):\n    n, h, w = a.shape\n    if h % cell or w % cell:\n        raise ValueError(f'tile size {h}x{w} is not a multiple of the cell size {cell}')\n    return a.reshape(n, h // cell, cell, w // cell, cell)\n\n\ndef cell_table(ids, y3, p_weed, cell):\n    \"\"\"One row per cell: tile id, acquisition date, true weed and crop fractions, and the score s.\"\"\"\n    ids = np.asarray(ids)\n    if len(ids) != len(y3) or y3.shape != p_weed.shape:\n        raise ValueError(f'misaligned inputs: ids {len(ids)}, labels {y3.shape}, probabilities {p_weed.shape}')\n    w = cell_view(y3 == 2, cell).mean(axis=(2, 4))\n    crop = cell_view(y3 == 1, cell).mean(axis=(2, 4))\n    # float64: a float32 mean turns an exact 1 % cover (400 / 40000) into 0.0099999998 and flips a decision at tau\n    s = cell_view(p_weed.astype(np.float64), cell).mean(axis=(2, 4))\n    tile = np.repeat(np.arange(len(ids)), w.shape[1] * w.shape[2])\n    return pd.DataFrame({'tile': ids[tile], 'date': [t[:10] for t in ids[tile]], 'cell_size': cell,\n                         'w': w.ravel(), 'crop': crop.ravel(), 's': s.ravel()})\n\n\ndef lower_threshold(pos_scores, alpha):\n    \"\"\"floor(alpha (n+1))-th smallest score of calibration cells that need spraying; -inf when there are too few,\n    so that 'need' is never ruled out.\"\"\"\n    s = np.sort(np.asarray(pos_scores, dtype=float))\n    k = math.floor(alpha * (len(s) + 1) + _EPS)\n    return float(s[k - 1]) if k >= 1 else -np.inf\n\n\ndef upper_threshold(neg_scores, beta):\n    \"\"\"ceil((1-beta)(n+1))-th smallest score of calibration cells that do not need spraying; +inf when there are\n    too few, so that 'no need' is never ruled out.\"\"\"\n    s = np.sort(np.asarray(neg_scores, dtype=float))\n    k = math.ceil((1 - beta) * (len(s) + 1) - _EPS)\n    return float(s[k - 1]) if k <= len(s) else np.inf\n\n\ndef decide_conformal(s, lam_lo, lam_hi):\n    s = np.asarray(s, dtype=float)\n    need_possible = s >= lam_lo\n    no_need_possible = s <= lam_hi\n    d = np.full(s.shape, DEFER)\n    d[need_possible & ~no_need_possible] = SPRAY\n    d[~need_possible & no_need_possible] = NO_SPRAY\n    return d  # both possible, or neither (lam_hi < s < lam_lo): DEFER\n\n\ndef decide_threshold(s, t):\n    return np.where(np.asarray(s, dtype=float) >= t, SPRAY, NO_SPRAY)\n\n\ndef decision_metrics(d, need, w, crop):\n    d, need, w, crop = map(np.asarray, (d, need, w, crop))\n    spray, defer, no_spray = d == SPRAY, d == DEFER, d == NO_SPRAY\n    effective = spray | (defer & need)  # deferred cells are scouted and sprayed only if needed\n\n    def share(mask, among=None):\n        if among is None:\n            return float(mask.mean()) if mask.size else math.nan\n        return float(mask[among].mean()) if among.any() else math.nan\n\n    return {\n        'n_cells': int(d.size),\n        'infestation': share(need),\n        'sprayed': share(spray),\n        'deferred': share(defer),\n        'effective_sprayed': share(effective),\n        'spray_reduction': 1 - share(effective),\n        'missed_need_rate': share(no_spray, need),       # P(NO_SPRAY | needs spraying)\n        'false_spray_rate': share(spray, ~need),         # P(SPRAY | does not need spraying)\n        'missed_weed_area': float(w[no_spray].sum() / w.sum()) if w.sum() > 0 else math.nan,\n        'crop_area_sprayed': float(crop[spray].sum() / crop.sum()) if crop.sum() > 0 else math.nan,\n        'spray_surplus': share(spray) - share(need),     # Melki et al. 2025, eq. 13 (before scouting)\n    }\n\n\ndef policy_decisions(cal, test, tau, alpha, beta):\n    \"\"\"Thresholds from `cal` cells, decisions of every policy on `test` cells (DataFrames from cell_table).\"\"\"\n    need_cal = cal['w'].to_numpy() >= tau\n    s_cal = cal['s'].to_numpy()\n    lam_lo = lower_threshold(s_cal[need_cal], alpha)\n    lam_hi = upper_threshold(s_cal[~need_cal], beta)\n    s = test['s'].to_numpy()\n    need = test['w'].to_numpy() >= tau\n    policies = {\n        'broadcast': np.full(s.shape, SPRAY),\n        'oracle': np.where(need, SPRAY, NO_SPRAY),\n        'plug_in': decide_threshold(s, tau),                 # spray if the predicted weed fraction reaches tau\n        'conformal_miss_only': decide_threshold(s, lam_lo),  # spray unless 'need' is ruled out (no deferral)\n        'conformal_3way': decide_conformal(s, lam_lo, lam_hi),\n    }\n    info = {'lam_lo': lam_lo, 'lam_hi': lam_hi, 'n_cal_need': int(need_cal.sum()),\n            'n_cal_no_need': int((~need_cal).sum())}\n    return policies, info\n\n\ndef evaluate_policies(cal, test, tau, alpha, beta):\n    \"\"\"Calibrate on `cal` cells and evaluate every policy on `test` cells.\"\"\"\n    policies, info = policy_decisions(cal, test, tau, alpha, beta)\n    need = test['w'].to_numpy() >= tau\n    w, crop = test['w'].to_numpy(), test['crop'].to_numpy()\n    return [{'policy': k, 'tau': tau, 'alpha': alpha, 'beta': beta, **info, **decision_metrics(d, need, w, crop)}\n            for k, d in policies.items()]\n\n\ndef crossfit_decisions(cells, fold, tau, alpha, beta):\n    \"\"\"Decide each fold with thresholds calibrated on the other folds, so calibration cells never come from the\n    evaluated locations. `fold` gives one fold label per row of `cells`. Returns None when there is only one fold,\n    else (policies, per_row, per_fold): policies maps each policy to decisions aligned with the rows of `cells`;\n    per_row has the lam_lo / lam_hi applied to each row; per_fold lists policy_decisions' info in sorted fold order.\"\"\"\n    fold = np.asarray(fold)\n    folds = np.unique(fold)\n    if len(folds) < 2:\n        return None  # nothing left to calibrate on (e.g. a date whose tiles all fall in one fold)\n    policies, per_fold = {}, []\n    per_row = {'lam_lo': np.empty(len(cells)), 'lam_hi': np.empty(len(cells))}\n    for k in folds:\n        ev = fold == k\n        pol, info = policy_decisions(cells[~ev], cells[ev], tau, alpha, beta)\n        for name, d in pol.items():\n            policies.setdefault(name, np.empty(len(cells), dtype=int))[ev] = d\n        per_row['lam_lo'][ev], per_row['lam_hi'][ev] = info['lam_lo'], info['lam_hi']\n        per_fold.append(info)\n    return policies, per_row, per_fold\n\n\ndef evaluate_crossfit(cells, fold, tau, alpha, beta):\n    \"\"\"Every policy decided by crossfit_decisions, then all cells scored together. lam_lo / lam_hi are tuples of the\n    thresholds applied to each fold, in sorted fold order.\"\"\"\n    res = crossfit_decisions(cells, fold, tau, alpha, beta)\n    if res is None:\n        return []\n    policies, _, per_fold = res\n    need = cells['w'].to_numpy() >= tau\n    w, crop = cells['w'].to_numpy(), cells['crop'].to_numpy()\n    info = {'lam_lo': tuple(i['lam_lo'] for i in per_fold), 'lam_hi': tuple(i['lam_hi'] for i in per_fold),\n            'n_cal_need': min(i['n_cal_need'] for i in per_fold),\n            'n_cal_no_need': min(i['n_cal_no_need'] for i in per_fold)}\n    return [{'policy': name, 'tau': tau, 'alpha': alpha, 'beta': beta, **info, **decision_metrics(d, need, w, crop)}\n            for name, d in policies.items()]\n", "src/weeds/analyze.py": "\"\"\"Pilot decision analysis and go / no-go check (numpy and pandas only; runs on Colab or locally).\n\n  python -m src.weeds.analyze --data <dir with weedsgalore_<split>.npz> --out <dir with probs_*.npz, segmentation_metrics.json>\n\nThe validation split already chose the model (best epoch, early stopping), so it is not independent of the model.\nPrimary calibration is therefore 2-fold on the test split by field location: thresholds from 4 locations decide the\nother 4, and vice versa. Calibration on the validation split is reported as a secondary variant.\nThe weed-cover thresholds tau are illustrative until replaced by sourced action thresholds.\n\"\"\"\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom src.weeds.data import (DATES, EXPECTED_TEST_LOCATIONS, PAPER_TEST_3CLASS_MSI, PRE_HERBICIDE_DATES,\n                            load_npz, location_blocks, to_3class)\nfrom src.weeds.decision import cell_table, evaluate_crossfit, evaluate_policies\n\nCELL_SIZES = (100, 200)                # 0.25 m and 0.5 m cells at 2.5 mm GSD\nTAUS = (0.001, 0.005, 0.01, 0.02)      # illustrative weed-cover fractions that make a cell need spraying\nALPHA = BETA = 0.1\nCAL_TEST_2FOLD = 'test 2-fold by location'\nCAL_VAL = 'val (also used for model selection)'\nPRIMARY = {'cell_size': 200, 'tau': 0.01, 'policy': 'conformal_3way', 'scope': 'pre-herbicide pooled',\n           'calibration': CAL_TEST_2FOLD}\n# Proposed pilot criteria (agreed before running; adjustable). C1: test mIoU at most 5 points below the paper's value\n# for the same input; segmentation_metrics.json of the authors' checkpoints carries it as 'paper_test', our own runs\n# (5-band input) are compared with Table 3's MSI value.\nCRITERIA = {'miou_below_paper_max': 5.0, 'spray_reduction_min': 0.30, 'deferred_max': 0.30}\n\n\ndef load_cells(data_dir, out_dir, split, cell):\n    d = load_npz(Path(data_dir) / f'weedsgalore_{split}.npz')\n    with np.load(Path(out_dir) / f'probs_{split}.npz') as p:\n        ids, p_weed = p['ids'].astype(str), p['p_weed']\n    n = len(ids)\n    if not np.array_equal(ids, d['ids'][:n]):\n        raise ValueError(f'{split}: probabilities and labels are not in the same tile order')\n    return cell_table(ids, to_3class(d['y'][:n]), p_weed, cell)\n\n\ndef test_location_folds(test_ids):\n    \"\"\"Fold 0 / 1 per test tile, alternating over the 8 field locations (4 locations per fold).\"\"\"\n    blocks = location_blocks(test_ids)\n    if len(blocks) != EXPECTED_TEST_LOCATIONS:\n        raise ValueError(f'expected {EXPECTED_TEST_LOCATIONS} test locations, found {len(blocks)}')\n    return {t: i % 2 for i, b in enumerate(blocks) for t in b}\n\n\ndef run(data_dir, out_dir):\n    rows = []\n    scopes = [('pre-herbicide pooled', PRE_HERBICIDE_DATES)] + [(d, (d,)) for d in DATES]\n    for cell in CELL_SIZES:\n        val_all, test_all = load_cells(data_dir, out_dir, 'val', cell), load_cells(data_dir, out_dir, 'test', cell)\n        fold_of = test_location_folds(list(dict.fromkeys(test_all['tile'])))\n        for tau in TAUS:\n            for scope, dates in scopes:\n                t = test_all[test_all['date'].isin(dates)]\n                if t.empty:\n                    continue\n                base = {'cell_size': cell, 'scope': scope}\n                rows += [{**base, 'calibration': CAL_TEST_2FOLD, **r}\n                         for r in evaluate_crossfit(t, t['tile'].map(fold_of).to_numpy(), tau, ALPHA, BETA)]\n                v = val_all[val_all['date'].isin(dates)]\n                if not v.empty:\n                    rows += [{**base, 'calibration': CAL_VAL, **r} for r in evaluate_policies(v, t, tau, ALPHA, BETA)]\n    return pd.DataFrame(rows)\n\n\ndef pick(df, **keys):\n    m = np.ones(len(df), dtype=bool)\n    for k, v in keys.items():\n        m &= (df[k] == v).to_numpy()\n    if m.sum() != 1:\n        raise ValueError(f'expected exactly one row for {keys}, found {m.sum()}')\n    return df[m].iloc[0]\n\n\ndef go_no_go(df, seg):\n    key = {k: PRIMARY[k] for k in ('cell_size', 'tau', 'scope', 'calibration')}\n    r = pick(df, policy=PRIMARY['policy'], **key)\n    oracle = pick(df, policy='oracle', **key)\n    miou = seg['test']['miou']\n    miou_min = seg.get('paper_test', PAPER_TEST_3CLASS_MSI)['miou'] - CRITERIA['miou_below_paper_max']\n    checks = {\n        'C1_test_miou': {'value': miou, 'required': f'>= {miou_min:.2f}', 'pass': bool(miou >= miou_min)},\n        'C2_spray_reduction': {'value': r['spray_reduction'], 'required': f\">= {CRITERIA['spray_reduction_min']}\",\n                               'pass': bool(r['spray_reduction'] >= CRITERIA['spray_reduction_min'])},\n        'C3_deferred': {'value': r['deferred'], 'required': f\"<= {CRITERIA['deferred_max']}\",\n                        'pass': bool(r['deferred'] <= CRITERIA['deferred_max'])},\n    }\n    info = {\n        'primary_setting': PRIMARY, 'alpha': ALPHA, 'beta': BETA,\n        'missed_need_rate': r['missed_need_rate'], 'false_spray_rate': r['false_spray_rate'],\n        'infestation': r['infestation'], 'oracle_spray_reduction': oracle['spray_reduction'],\n        'c2_reachable': bool(oracle['spray_reduction'] >= CRITERIA['spray_reduction_min']),\n        'n_cal_need': int(r['n_cal_need']), 'n_cal_no_need': int(r['n_cal_no_need']),\n        'lam_lo': r['lam_lo'], 'lam_hi': r['lam_hi'],\n    }\n    return {'verdict': 'GO' if all(c['pass'] for c in checks.values()) else 'NO-GO', 'checks': checks, 'info': info}\n\n\ndef clean(o):\n    \"\"\"JSON-safe: NaN / inf -> None, numpy scalars -> Python.\"\"\"\n    if isinstance(o, dict):\n        return {k: clean(v) for k, v in o.items()}\n    if isinstance(o, (list, tuple)):\n        return [clean(v) for v in o]\n    if isinstance(o, (np.floating, float)):\n        return None if not math.isfinite(float(o)) else float(o)\n    if isinstance(o, np.integer):\n        return int(o)\n    if isinstance(o, np.bool_):\n        return bool(o)\n    return o\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--data', required=True)\n    ap.add_argument('--out', required=True)\n    args = ap.parse_args()\n    out = Path(args.out)\n    df = run(args.data, out)\n    df.to_csv(out / 'decision_metrics.csv', index=False)\n    seg = json.loads((out / 'segmentation_metrics.json').read_text())\n    verdict = go_no_go(df, seg)\n    (out / 'go_no_go.json').write_text(json.dumps(clean(verdict), indent=1))\n\n    cols = ['policy', 'infestation', 'sprayed', 'deferred', 'spray_reduction', 'missed_need_rate', 'false_spray_rate',\n            'missed_weed_area', 'crop_area_sprayed']\n    same = (df['cell_size'] == PRIMARY['cell_size']) & (df['tau'] == PRIMARY['tau'])\n    pd.set_option('display.width', 200)\n    for cal in (CAL_TEST_2FOLD, CAL_VAL):\n        print(f\"\\nCell {PRIMARY['cell_size']} px, tau {PRIMARY['tau']}, {PRIMARY['scope']}, calibration: {cal}, \"\n              f'alpha = beta = {ALPHA}')\n        print(df[same & (df['scope'] == PRIMARY['scope']) & (df['calibration'] == cal)][cols].round(3)\n              .to_string(index=False))\n    per_date = df[same & (df['policy'] == PRIMARY['policy']) & (df['scope'] != PRIMARY['scope'])]\n    print('\\nPer acquisition date (conformal_3way):')\n    print(per_date[['scope', 'calibration', 'n_cal_need', 'infestation', 'deferred', 'spray_reduction',\n                    'missed_need_rate', 'false_spray_rate']].round(3).to_string(index=False))\n    print('\\nGO / NO-GO:', verdict['verdict'])\n    for name, c in verdict['checks'].items():\n        print(f\"  {name}: {c['value']:.3f} (required {c['required']}) -> {'PASS' if c['pass'] else 'FAIL'}\")\n    i = verdict['info']\n    print(f\"  validity: missed-need rate {i['missed_need_rate']:.3f} vs alpha {ALPHA}, \"\n          f\"false-spray rate {i['false_spray_rate']:.3f} vs beta {BETA}\")\n    if not i['c2_reachable']:\n        print(f\"  note: C2 cannot be met at this tau even by the oracle (oracle reduction {i['oracle_spray_reduction']:.3f})\")\n\n\nif __name__ == '__main__':\n    main()\n", "src/weeds/authors.py": "\"\"\"Evaluate the DeepLabv3+ checkpoints released by the WeedsGalore authors (ckpts.zip on the dataset DOI page) with\nthe evaluation code of this project (src.weeds.train.evaluate, the function that scored our own runs). Two uses:\ncheck our evaluation against the paper (Celikkan et al. 2025, Tables 3 and 4), and run the decision layer on the\nauthors' own model next to ours.\n\nTheir network is VainF's DeepLabV3Plus with the first convolution replaced to take `in_channels` bands (their\nsrc/nets/deeplabv3plus and src/evaluate.py); src.weeds.model has the same layers under other names, so the state dict\nis renamed, never converted, and loaded with strict=True. Inputs are scaled to [0, 1] and RGB uses the first three\nbands (R, G, B), as in their dataset class. Each checkpoint is evaluated twice:\n  float32  as in the authors' evaluate.py: the numbers to compare with the paper;\n  float16  autocast, as our runs were evaluated: shows what float16 inference alone changes, and gives the\n           probabilities for the decision layer, so it treats both models alike.\nRun on Colab GPU (on CPU both modes are float32):\n\n  python -m src.weeds.authors --data <dir with weedsgalore_<split>.npz> --ckpts <dir with dlv3p_*.pth> --out <dir>\n\nFor each checkpoint <out>/<name>/ gets segmentation_metrics.json ('val' and 'test' from float16 inference, 'fp32' with\nboth splits in float32); 3-class checkpoints also get probs_val.npz and probs_test.npz in the format of\nsrc.weeds.train, so `python -m src.weeds.analyze --out <out>/<name>` works on them.\n\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom src.weeds.analyze import clean\nfrom src.weeds.data import BANDS, PAPER_TEST_3CLASS_MSI, load_npz\nfrom src.weeds.model import DeepLabV3Plus\nfrom src.weeds.train import evaluate\n\nCKPT_URL = 'https://doidata.gfz.de/weedsgalore_e_celikkan_2024/ckpts.zip'\nCKPT_ZIP_BYTES = 886908486  # size served by GFZ on 2026-11-05\n# deterministic checkpoints in ckpts.zip: name -> (input bands, classes); the dlv3p_do_* (MC dropout) are not used\nCKPTS = {'dlv3p_msi_3': (5, 3), 'dlv3p_rgb_3': (3, 3), 'dlv3p_msi_6': (5, 6), 'dlv3p_rgb_6': (3, 6)}\n# Celikkan et al. 2025, DeepLabv3+, test IoU in %: Table 3 (3 classes) and Table 4 (6 classes)\nPAPER = {\n    'dlv3p_msi_3': PAPER_TEST_3CLASS_MSI,\n    'dlv3p_rgb_3': {'miou': 79.33, 'background': 97.97, 'crop': 67.93, 'weed': 72.08},\n    'dlv3p_msi_6': {'miou': 55.52, 'background': 98.37, 'maize': 73.03, 'amaranth': 76.17, 'barnyard_grass': 53.55,\n                    'quickweed': 21.11, 'weed_other': 10.86},\n    'dlv3p_rgb_6': {'miou': 50.81, 'background': 97.94, 'maize': 71.46, 'amaranth': 74.32, 'barnyard_grass': 45.95,\n                    'quickweed': 6.26, 'weed_other': 8.92},\n}\n# their parameter-name prefixes -> ours (src.weeds.model)\nRENAMES = [('backbone.conv1.', 'stem.0.'), ('backbone.bn1.', 'stem.1.'), ('backbone.layer', 'layer'),\n           ('classifier.project.', 'low_proj.'), ('classifier.aspp.convs.', 'aspp.branches.'),\n           ('classifier.aspp.project.', 'aspp.project.'), ('classifier.classifier.', 'classifier.')]\n\n\ndef rename_state_dict(sd):\n    out = {}\n    for k, v in sd.items():\n        name = k[len('module.'):] if k.startswith('module.') else k\n        for old, new in RENAMES:\n            if name.startswith(old):\n                out[new + name[len(old):]] = v\n                break\n        else:\n            raise KeyError(f'unexpected parameter name in checkpoint: {k}')\n    return out\n\n\ndef load_checkpoint(path, bands, classes, dev):\n    # weights_only: a third-party file must not be able to run code when unpickled\n    sd = torch.load(path, map_location='cpu', weights_only=True)\n    model = DeepLabV3Plus(bands, classes, pretrained_backbone=False)\n    model.load_state_dict(rename_state_dict(sd), strict=True)\n    return model.to(dev).eval()\n\n\ndef per_class(res):\n    return {k: v for k, v in res.items() if k != 'confusion'}\n\n\ndef run(data_dir, ckpt_dir, out_dir, dev):\n    if BANDS[:3] != ('R', 'G', 'B'):\n        raise ValueError('RGB checkpoints expect the first three bands to be R, G, B')\n    missing = [n for n in CKPTS if not (Path(ckpt_dir) / f'{n}.pth').exists()]\n    if missing:\n        raise FileNotFoundError(f'checkpoints not found in {ckpt_dir}: {missing}')\n    splits = {s: load_npz(Path(data_dir) / f'weedsgalore_{s}.npz') for s in ('val', 'test')}\n    summary = {}\n    for name, (bands, classes) in CKPTS.items():\n        model = load_checkpoint(Path(ckpt_dir) / f'{name}.pth', bands, classes, dev)\n        out = Path(out_dir) / name\n        out.mkdir(parents=True, exist_ok=True)\n        report = {'fp32': {}}\n        for split, d in splits.items():\n            report['fp32'][split] = evaluate(model, d, dev, classes=classes, bands=bands, amp=False)\n            if classes == 3:\n                res, probs = evaluate(model, d, dev, keep_probs=True, classes=3, bands=bands, amp=True)\n                np.savez_compressed(out / f'probs_{split}.npz', ids=d['ids'], p_crop=probs[:, 0], p_weed=probs[:, 1])\n            else:\n                res = evaluate(model, d, dev, classes=classes, bands=bands, amp=True)\n            report[split] = res\n        paper = PAPER[name]\n        report['paper_test'] = paper\n        report['fp32_test_minus_paper_miou'] = report['fp32']['test']['miou'] - paper['miou']\n        report['test_minus_paper_miou'] = report['test']['miou'] - paper['miou']\n        (out / 'segmentation_metrics.json').write_text(json.dumps(clean(report), indent=1))\n        summary[name] = {'bands': bands, 'classes': classes, 'paper': paper,\n                         'fp32_test': per_class(report['fp32']['test']), 'fp16_test': per_class(report['test']),\n                         'fp32_test_minus_paper_miou': report['fp32_test_minus_paper_miou'],\n                         'fp16_test_minus_paper_miou': report['test_minus_paper_miou']}\n        t32, t16 = report['fp32']['test'], report['test']\n        print(f'{name}: test mIoU float32 {t32[\"miou\"]:.2f}, float16 {t16[\"miou\"]:.2f}, paper {paper[\"miou\"]:.2f} '\n              f'(float32 - paper {report[\"fp32_test_minus_paper_miou\"]:+.2f}) | float32 / paper: ' +\n              ', '.join(f'{c} {t32[c]:.2f} / {v:.2f}' for c, v in paper.items() if c != 'miou'))\n        del model\n    Path(out_dir).mkdir(parents=True, exist_ok=True)\n    (Path(out_dir) / 'authors_vs_paper.json').write_text(json.dumps(clean(summary), indent=1))\n    return summary\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--data', required=True)\n    ap.add_argument('--ckpts', required=True)\n    ap.add_argument('--out', required=True)\n    args = ap.parse_args()\n    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n    print('device:', dev, torch.cuda.get_device_name(0) if dev.type == 'cuda' else '')\n    run(args.data, args.ckpts, args.out, dev)\n\n\nif __name__ == '__main__':\n    main()\n", "src/weeds/runs.py": "\"\"\"Further training runs to explain the gap between our pilot run 2 (test mIoU 79.25, one seed) and the paper (82.90),\nwhose released checkpoint our evaluation reproduces exactly (src.weeds.authors). The authors' training code differs\nfrom ours only in its schedule: 100 epochs without early stopping, checkpoints chosen on validation. So the runs\nmeasure seed variance under both schedules, plus one variant with a decaying learning rate. All use batch 8 and the\n5-band input. notebooks/colab_weeds_runs.ipynb trains them in the order below, as many as fit in a Colab session,\nevaluates each on test, runs the decision layer, and resumes the rest in the next session.\n\"\"\"\nimport json\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom src.weeds.analyze import PRIMARY, pick\nfrom src.weeds.train import parser\n\nBATCH = 8\nRUNS = [  # in order of priority\n    # the authors' schedule (github.com/GFZ/weedsgalore README: --epochs 100), constant learning rate\n    {'name': 'e100_seed0', 'seed': 0, 'epochs': 100, 'patience': 100, 'lr_schedule': 'constant'},\n    {'name': 'e100_seed1', 'seed': 1, 'epochs': 100, 'patience': 100, 'lr_schedule': 'constant'},\n    {'name': 'e100_seed2', 'seed': 2, 'epochs': 100, 'patience': 100, 'lr_schedule': 'constant'},\n    # 100 epochs with the learning rate decayed to zero: a steadier end of training (not in the authors' code)\n    {'name': 'e100_cosine_seed0', 'seed': 0, 'epochs': 100, 'patience': 100, 'lr_schedule': 'cosine'},\n    # our pilot schedule (run 2 = seed 0): early stopping after 30 epochs without improvement, at most 200\n    {'name': 'es_seed1', 'seed': 1, 'epochs': 200, 'patience': 30, 'lr_schedule': 'constant'},\n    {'name': 'es_seed2', 'seed': 2, 'epochs': 200, 'patience': 30, 'lr_schedule': 'constant'},\n]\n\n\ndef train_args(run):\n    \"\"\"Command-line arguments of src.weeds.train for one run (data, output and budget are added by the caller).\"\"\"\n    return ['--batch', str(BATCH), '--seed', str(run['seed']), '--epochs', str(run['epochs']),\n            '--patience', str(run['patience']), '--lr-schedule', run['lr_schedule']]\n\n\ndef schedule(done):\n    \"\"\"Schedule label from a train_done.json; runs written before these fields existed used the parser defaults.\"\"\"\n    p = parser()\n    epochs, patience = done.get('epochs', p.get_default('epochs')), done.get('patience', p.get_default('patience'))\n    lr = done.get('lr_schedule', p.get_default('lr_schedule'))\n    if lr != 'constant':\n        return f'{epochs} epochs, {lr} lr'\n    return f'{epochs} epochs fixed' if patience >= epochs else f'early stopping ({patience})'\n\n\ndef summarize(run_dirs):\n    \"\"\"One row per finished run (train_done, segmentation_metrics, decision_metrics present): schedule, seed, best\n    epoch, validation and test mIoU, and the decisions of conformal_3way in the primary setting.\"\"\"\n    rows = []\n    for name, d in run_dirs.items():\n        d = Path(d)\n        files = [d / f for f in ('train_done.json', 'best.json', 'segmentation_metrics.json', 'decision_metrics.csv')]\n        if not all(f.exists() for f in files):\n            continue\n        done, best, seg = (json.loads(f.read_text()) for f in files[:3])\n        r = pick(pd.read_csv(files[3]), policy=PRIMARY['policy'],\n                 **{k: PRIMARY[k] for k in ('cell_size', 'tau', 'scope', 'calibration')})\n        rows.append({'run': name, 'schedule': schedule(done), 'seed': done['seed'], 'epochs_run': done['epochs_run'],\n                     'best_epoch': best['epoch'], 'val_miou': best['val_miou'], 'test_miou': seg['test']['miou'],\n                     'test_crop': seg['test']['crop'], 'test_weed': seg['test']['weed'],\n                     'spray_reduction': r['spray_reduction'], 'deferred': r['deferred'],\n                     'missed_need_rate': r['missed_need_rate'], 'false_spray_rate': r['false_spray_rate']})\n    return pd.DataFrame(rows)\n\n\ndef by_schedule(df):\n    \"\"\"Count, mean, sample standard deviation (NaN for a single run) and range of test mIoU per schedule.\"\"\"\n    return (df.groupby('schedule')['test_miou'].agg(n='count', mean='mean', std='std', min='min', max='max')\n            .reset_index())\n"}
for rel, text in FILES.items():
    path = WORK / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print('Đã ghi', len(FILES), 'file mã nguồn vào', WORK)
print('Batch', BATCH, '| thư mục kết quả:', RUN)

import subprocess
def run(*args):
    """Run a module of the pilot, stream its output, and stop 'Run all' if it fails."""
    p = subprocess.Popen([sys.executable, '-u', '-m', *map(str, args)], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}). Dừng lại và gửi thông báo lỗi ở trên cho Claude.')

## 2. Phần A: tải và chuẩn bị dữ liệu (runtime CPU)

In [ ]:
import subprocess
from src.weeds.data import build_npz

ZIP = DATA_DRIVE / 'weedsgalore-dataset.zip'
if not ZIP.exists() or ZIP.stat().st_size != 336989804:
    subprocess.run(['curl', '-L', '--retry', '3', '-o', str(ZIP), 'https://doidata.gfz.de/weedsgalore_e_celikkan_2024/weedsgalore-dataset.zip'], check=True)
assert ZIP.stat().st_size == 336989804, f'Kích thước file zip không đúng: {ZIP.stat().st_size}'

if all((DATA_DRIVE / f'weedsgalore_{s}.npz').exists() for s in ('train', 'val', 'test')):
    print('Dữ liệu gọn đã có trên Drive, bỏ qua.')
else:
    build_npz(ZIP, DATA_DRIVE)

## 3. Phần B: kiểm tra GPU và chép dữ liệu về máy Colab

Trước mục này: *Runtime → Change runtime type → T4 GPU*, rồi chạy lại mục 1.

In [ ]:
import shutil, torch
!nvidia-smi -L
print('PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chưa có GPU: đổi runtime sang T4 rồi chạy lại mục 1.'
DATA = Path('/content/data')
DATA.mkdir(exist_ok=True)
for s in ('train', 'val', 'test'):
    src = DATA_DRIVE / f'weedsgalore_{s}.npz'
    assert src.exists(), f'Chưa có {src}: chạy Phần A (mục 2) trước.'
    if not (DATA / src.name).exists():
        shutil.copy(src, DATA)
print(sorted(p.name for p in DATA.iterdir()))

## 4. Chạy thử (khoảng vài phút)

Chạy 1 epoch chỉ 2 vòng lặp, rồi đánh giá trên 2 ô ảnh. Mục đích là bắt lỗi và kiểm tra bộ nhớ GPU (dòng `peak GPU`) ở **cùng batch 8** với lần huấn luyện thật. Bộ nhớ ước tính khoảng 9–10 GB, T4 có khoảng 15 GB. Kết quả chạy thử để ở `/content/smoke`, không dùng để báo cáo. Nếu báo lỗi hết bộ nhớ (out of memory), dừng lại và báo cho Claude.

In [ ]:
shutil.rmtree('/content/smoke', ignore_errors=True)
run('src.weeds.train', '--data', DATA, '--out', '/content/smoke', '--smoke', '--batch', BATCH)

## 5. Huấn luyện đầy đủ

Tối đa 200 epoch, dừng sớm nếu mIoU trên val không tăng sau 30 epoch, tự dừng trước 3,5 giờ. Mỗi epoch in ra một dòng gồm loss, mIoU val, thời gian và bộ nhớ GPU. Nếu thấy dòng `time budget ... reached`, ở phiên sau chạy lại mục 1, 3 rồi mục này.

In [ ]:
run('src.weeds.train', '--stage', 'train', '--data', DATA, '--out', RUN, '--batch', BATCH)

## 6. Đánh giá trên tập test và lưu xác suất dự đoán (chỉ chạy khi huấn luyện đã xong)

In [ ]:
done = RUN / 'train_done.json'
if done.exists():
    print(done.read_text())
    run('src.weeds.train', '--stage', 'test', '--data', DATA, '--out', RUN)
else:
    print('Huấn luyện chưa xong: ở phiên sau chạy lại mục 1, 3 rồi 5.')

## 7. Tầng quyết định và kết luận đạt/không đạt

In [ ]:
if (RUN / 'probs_test.npz').exists():
    run('src.weeds.analyze', '--data', DATA, '--out', RUN)
else:
    print('Chưa có xác suất dự đoán: chạy mục 6 trước.')

## 8. Đóng gói kết quả (không kèm checkpoint mô hình)

In [ ]:
import zipfile
ZIP_OUT = BASE / f'ket_qua_weeds_pilot_b{BATCH}.zip'
with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RUN.iterdir()):
        if f.suffix in ('.json', '.csv', '.npz'):
            z.write(f, RESULTS / f.name)
print('Đã tạo', ZIP_OUT, f'({ZIP_OUT.stat().st_size / 1e6:.1f} MB)')

**Bước tiếp theo:** tải `ket_qua_weeds_pilot_b8.zip` về, giải nén vào thư mục gốc của repo (sẽ tạo thư mục `results/weeds_pilot_b8/`, không đụng tới kết quả lần 1 trong `results/weeds_pilot/`), rồi nhờ Claude phân tích.